# 02 — Modelado estrella (Spark)

**Fase 4.** Lee `/movilidad/refined/` y materializa el modelo dimensional en `/movilidad/modeled/`.

Diagrama de referencia: `plan/03-modelo-estrella.md`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StructType, StructField

spark = (
    SparkSession.builder
    .appName("movilidad-modelo-estrella")
    .getOrCreate()
)

REFINED = "hdfs:///movilidad/refined"
MODELED = "hdfs:///movilidad/modeled"
INT = IntegerType()

spark.sparkContext.setLogLevel("WARN")


def guardar_modeled(df, nombre, int_cols=None):
    """Guarda Parquet con columnas INT explícitas (Hive espera int, no int64)."""
    if int_cols:
        for col in int_cols:
            if col in df.columns:
                df = df.withColumn(col, F.col(col).cast(INT))
    path = f"{MODELED}/{nombre}"
    df.write.mode("overwrite").parquet(path)
    print(f"Guardado: {path}  ({df.count():,} filas)")

In [ ]:
tipos = spark.read.parquet(f"{REFINED}/tipos_vehiculos")
parque_2019 = spark.read.parquet(f"{REFINED}/parque_2019")
parque_2024 = spark.read.parquet(f"{REFINED}/parque_2024")
sensores_2019 = spark.read.parquet(f"{REFINED}/sensores_2019")
sensores_2024 = spark.read.parquet(f"{REFINED}/sensores_2024")
dim_ubicacion = spark.read.parquet(f"{REFINED}/dim_ubicacion")
conteo_2019 = spark.read.parquet(f"{REFINED}/conteo_2019")
conteo_2024 = spark.read.parquet(f"{REFINED}/conteo_2024")
velocidad_2019 = spark.read.parquet(f"{REFINED}/velocidad_2019")
velocidad_2024 = spark.read.parquet(f"{REFINED}/velocidad_2024")

parque_all = parque_2019.unionByName(parque_2024)
conteo_all = conteo_2019.unionByName(conteo_2024)
velocidad_all = velocidad_2019.unionByName(velocidad_2024)

print("Refined cargado OK")

## 1. Dimensiones

In [ ]:
dim_ubicacion_out = dim_ubicacion.select(
    "id_ubicacion", "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente",
    "latitud", "longitud", "tramo_descripcion"
)
guardar_modeled(dim_ubicacion_out, "dim_ubicacion", int_cols=["id_ubicacion"])

In [ ]:
dim_anio_parque = spark.createDataFrame(
    [(2019,), (2024,)],
    schema=StructType([StructField("anio_referencia", IntegerType(), False)]),
).withColumn("id_anio_parque", F.col("anio_referencia").cast(INT))

guardar_modeled(dim_anio_parque, "dim_anio_parque", int_cols=["anio_referencia", "id_anio_parque"])

In [ ]:
dim_combustible = (
    parque_all.select("combustible_norm", "categoria_combustible").distinct()
    .withColumnRenamed("combustible_norm", "combustible_original")
    .withColumn(
        "id_combustible",
        F.abs(F.hash("combustible_original", "categoria_combustible")).cast(INT)
    )
)
guardar_modeled(dim_combustible, "dim_combustible", int_cols=["id_combustible"])

In [ ]:
dim_tipo_vehiculo = (
    tipos.select("cod_tipveh", "tipo_vehiculo", "tipo_vehiculo_limpio", "es_tipo_valido")
    .withColumn("id_tipo_vehiculo", F.col("cod_tipveh").cast(INT))
)
guardar_modeled(dim_tipo_vehiculo, "dim_tipo_vehiculo", int_cols=["cod_tipveh", "id_tipo_vehiculo"])

In [ ]:
def base_tiempo(df):
    return df.select("fecha", "hora", "anio_referencia").distinct()

tiempo_raw = (
    base_tiempo(conteo_all)
    .unionByName(base_tiempo(velocidad_all))
    .distinct()
)

dim_tiempo = (
    tiempo_raw
    .withColumn("fecha", F.to_date("fecha"))
    .withColumn("hora_del_dia", F.substring("hora", 1, 2).cast(IntegerType()))
    .withColumn("minuto", F.substring("hora", 4, 2).cast(IntegerType()))
    .withColumn("dia_semana", F.date_format("fecha", "EEEE"))
    .withColumn(
        "franja_horaria",
        F.when(F.col("hora_del_dia").between(0, 6), "MADRUGADA")
         .when(F.col("hora_del_dia").between(7, 9), "PICO_MANANA")
         .when(F.col("hora_del_dia").between(10, 16), "MEDIODIA")
         .when(F.col("hora_del_dia").between(17, 19), "PICO_TARDE")
         .otherwise("NOCHE")
    )
    .withColumn(
        "es_hora_pico",
        F.col("hora_del_dia").between(7, 9) | F.col("hora_del_dia").between(17, 19)
    )
    .withColumn("id_tiempo", F.abs(F.hash("fecha", "hora", "anio_referencia")).cast(INT))
)

guardar_modeled(
    dim_tiempo, "dim_tiempo",
    int_cols=["anio_referencia", "hora_del_dia", "minuto", "id_tiempo"]
)

In [ ]:
def base_detector(df):
    return df.select(
        F.col("id_detector").alias("id_detector_natural"),
        "id_carril", "anio_referencia"
    ).distinct()

dim_detector = (
    base_detector(conteo_all)
    .unionByName(base_detector(velocidad_all))
    .distinct()
    .withColumn(
        "id_detector_surrogate",
        F.abs(F.hash("id_detector_natural", "id_carril", "anio_referencia")).cast(INT)
    )
)

guardar_modeled(
    dim_detector, "dim_detector",
    int_cols=["id_detector_natural", "id_carril", "anio_referencia", "id_detector_surrogate"]
)

In [ ]:
ubicacion_key = ["dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"]

dim_sensor = (
    sensores_2019.unionByName(sensores_2024)
    .join(dim_ubicacion_out.select("id_ubicacion", *ubicacion_key), ubicacion_key, "left")
    .withColumn("activo", F.lit(True))
    .withColumn(
        "id_sensor",
        F.abs(F.hash("dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud", "anio_referencia")).cast(INT)
    )
    .select("id_sensor", "id_ubicacion", "anio_referencia", "activo",
            "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud")
)

guardar_modeled(dim_sensor, "dim_sensor", int_cols=["id_sensor", "id_ubicacion", "anio_referencia"])

## 2. Tablas de hechos

In [ ]:
fact_parque = (
    parque_all.alias("p")
    .join(dim_combustible.alias("c"),
          (F.col("p.combustible_norm") == F.col("c.combustible_original")) &
          (F.col("p.categoria_combustible") == F.col("c.categoria_combustible")), "left")
    .join(dim_tipo_vehiculo.alias("t"), F.col("p.cod_tipveh") == F.col("t.cod_tipveh"), "left")
    .join(dim_anio_parque.alias("a"), F.col("p.anio_referencia") == F.col("a.anio_referencia"), "left")
    .select(
        F.monotonically_increasing_id().alias("id_parque"),
        F.col("c.id_combustible"),
        F.col("t.id_tipo_vehiculo"),
        F.col("a.id_anio_parque"),
        F.col("p.anio").alias("anio_vehiculo"),
        F.col("p.marca"), F.col("p.modelo"),
        F.col("p.cantidad"), F.col("p.es_tipo_valido")
    )
)

guardar_modeled(
    fact_parque, "fact_parque",
    int_cols=["id_combustible", "id_tipo_vehiculo", "id_anio_parque", "anio_vehiculo", "cantidad"]
)

In [ ]:
def armar_fact_conteo(df):
    return (
        df.alias("f")
        .join(dim_tiempo.alias("t"), ["fecha", "hora", "anio_referencia"], "left")
        .join(
            dim_detector.alias("d"),
            (F.col("f.id_detector") == F.col("d.id_detector_natural")) &
            (F.col("f.id_carril") == F.col("d.id_carril")) &
            (F.col("f.anio_referencia") == F.col("d.anio_referencia")),
            "left"
        )
        .select(
            F.monotonically_increasing_id().alias("id_conteo"),
            F.col("t.id_tiempo"),
            F.col("f.id_ubicacion"),
            F.col("d.id_detector_surrogate").alias("id_detector"),
            F.col("f.volumen"),
            F.col("f.volumen_hora"),
        )
    )


fact_conteo = armar_fact_conteo(conteo_all)
guardar_modeled(fact_conteo, "fact_conteo", int_cols=["id_tiempo", "id_ubicacion", "id_detector", "volumen"])

In [ ]:
def armar_fact_velocidad(df):
    return (
        df.alias("f")
        .join(dim_tiempo.alias("t"), ["fecha", "hora", "anio_referencia"], "left")
        .join(
            dim_detector.alias("d"),
            (F.col("f.id_detector") == F.col("d.id_detector_natural")) &
            (F.col("f.id_carril") == F.col("d.id_carril")) &
            (F.col("f.anio_referencia") == F.col("d.anio_referencia")),
            "left"
        )
        .select(
            F.monotonically_increasing_id().alias("id_velocidad"),
            F.col("t.id_tiempo"),
            F.col("f.id_ubicacion"),
            F.col("d.id_detector_surrogate").alias("id_detector"),
            F.col("f.velocidad_promedio"),
        )
    )


fact_velocidad = armar_fact_velocidad(velocidad_all)
guardar_modeled(fact_velocidad, "fact_velocidad", int_cols=["id_tiempo", "id_ubicacion", "id_detector", "velocidad_promedio"])

## 3. Validaciones del modelo

In [ ]:
print("=== Conteos del modelo estrella ===")
for tabla in [
    "dim_ubicacion", "dim_tiempo", "dim_detector", "dim_combustible",
    "dim_tipo_vehiculo", "dim_anio_parque", "dim_sensor",
    "fact_parque", "fact_conteo", "fact_velocidad",
]:
    n = spark.read.parquet(f"{MODELED}/{tabla}").count()
    print(f"  {tabla}: {n:,}")

fc = spark.read.parquet(f"{MODELED}/fact_conteo")
fv = spark.read.parquet(f"{MODELED}/fact_velocidad")
fp = spark.read.parquet(f"{MODELED}/fact_parque")

print(f"\nFK nulos fact_conteo — id_tiempo: {fc.filter(F.col('id_tiempo').isNull()).count()}")
print(f"FK nulos fact_conteo — id_ubicacion: {fc.filter(F.col('id_ubicacion').isNull()).count()}")
print(f"FK nulos fact_velocidad — id_tiempo: {fv.filter(F.col('id_tiempo').isNull()).count()}")
print(f"Total vehiculos fact_parque: {fp.agg(F.sum('cantidad')).collect()[0][0]:,}")

print("\nModeled en HDFS:")
!hdfs dfs -ls hdfs:///movilidad/modeled